In [1]:
"""
03_merge_team_valuation.py

Purpose:
Merge the avg_team_valuation_bil column into the final NBA ratings/all-star dataset.

Inputs:
- data/intermediate/ratings_final_allstar_indicators.csv
- data/intermediate/team_valuation_game_level.csv

Output:
- data/final/qss20_finalds.csv
"""

import pandas as pd
from pathlib import Path


# -----------------------------
# Functions
# -----------------------------

def load_csv(path):
    """Load a CSV file and print basic diagnostics."""
    df = pd.read_csv(path)
    print(f"Loaded {path}")
    print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
    return df


def check_required_columns(df, required_cols, df_name):
    """Make sure required columns exist before merging."""
    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise ValueError(
            f"{df_name} is missing required columns: {missing_cols}"
        )


def merge_team_valuation(final_df, valuation_df):
    """
    Merge avg_team_valuation_bil into the final dataset.

    The merge is done by GAME_ID because each row represents one NBA game.
    """

    check_required_columns(
        final_df,
        ["GAME_ID"],
        "Final dataset"
    )

    check_required_columns(
        valuation_df,
        ["GAME_ID", "avg_team_valuation_bil"],
        "Team valuation dataset"
    )

    # Keep only the merge key and valuation column
    valuation_df = valuation_df[
        ["GAME_ID", "avg_team_valuation_bil"]
    ].drop_duplicates()

    print("\nBefore merge:")
    print(f"Final dataset rows: {final_df.shape[0]}")
    print(f"Valuation dataset rows: {valuation_df.shape[0]}")
    print(
        "Final dataset games missing valuation before merge:",
        final_df["GAME_ID"].isna().sum()
    )

    merged_df = final_df.merge(
        valuation_df,
        on="GAME_ID",
        how="left",
        validate="one_to_one"
    )

    print("\nAfter merge:")
    print(f"Merged dataset rows: {merged_df.shape[0]}")
    print(f"Merged dataset columns: {merged_df.shape[1]}")
    print(
        "Rows missing avg_team_valuation_bil:",
        merged_df["avg_team_valuation_bil"].isna().sum()
    )

    return merged_df


# -----------------------------
# Main script
# -----------------------------

def main():
    # Relative paths for GitHub/repo compatibility
    data_dir = Path("data")
    intermediate_dir = data_dir / "intermediate"
    final_dir = data_dir / "final"

    final_dir.mkdir(parents=True, exist_ok=True)

    final_dataset_path = intermediate_dir / "ratings_final_allstar_indicators.csv"
    valuation_dataset_path = intermediate_dir / "team_valuation_game_level.csv"
    output_path = final_dir / "qss20_finalds.csv"

    final_df = load_csv(final_dataset_path)
    valuation_df = load_csv(valuation_dataset_path)

    merged_df = merge_team_valuation(final_df, valuation_df)

    merged_df.to_csv(output_path, index=False)

    print(f"\nSaved merged dataset to: {output_path}")


if __name__ == "__main__":
    main()

FileNotFoundError: [Errno 2] No such file or directory: 'data/intermediate/ratings_final_allstar_indicators.csv'